**Define the state space, action space, reward structure, and discount factor**

The Markov Decision Process (MDP) is defined using a 4×4 grid world. Each state represents a position in the grid and is denoted as a tuple (row, column), resulting in a total of 16 possible states. At each state, the agent can take one of four possible actions: up (U), down (D), left (L), or right (R). The transition function is deterministic, meaning that each action moves the agent in the intended direction. If the action would take the agent outside the grid boundaries, the agent remains in the same state. The reward function assigns a value of +1 for reaching the goal state, −1 for entering a hole (a negative terminal state), and −0.04 for all other states to encourage shorter paths. A discount factor of γ = 0.9 is used initially, which determines how much the agent values future rewards compared to immediate ones. The objective of this MDP is to compute the optimal value function and the corresponding optimal policy using the value iteration algorithm.

In [10]:
# Implement either value iteration or policy iteration and compute the final policy

import numpy as np

# grid size
rows, cols = 4, 4

# define actions
# up, down, left, right
actions = ['U', 'D', 'L', 'R']

# define how actions move agent
action_moves = {
    'U': (-1, 0), # move up one row
    'D': (1, 0), # move down one row
    'L': (0, -1), # move left one column
    'R': (0, 1) # move right one column
}

# initialize all states with small negative reward
reward_grid = np.full((rows, cols), -0.04)

# special states
goal_state = (3, 3) # positive reward
hole_state = (1, 1) # negative reward

# assign rewards to special states
reward_grid[goal_state] = 1 # reward for reaching goal
reward_grid[hole_state] = -1 # penalty for falling in hole

# discount factor - how much future rewards matter
gamma = 0.9

# convergence threshold - stop value iteration
theta = 1e-4

# initialize value function for all states
V = np.zeros((rows, cols))

'''
Checks whether given state is within grid boundreis

Parameters:
  state: position (row, col)

Returns:
  bool: true if state inside grid, false otherwise
'''
def is_valid(state):
    r, c = state
    return 0 <= r < rows and 0 <= c < cols

'''
Compute next state given current state and action
If action moves agent outside grid, agent remains in same state

Parameters:
  state: current position (row, col)
  action: one of U, D, L, R

Returns:
  tuple: next state (row, col)
'''
def get_next_state(state, action):
    move = action_moves[action]
    next_state = (state[0] + move[0], state[1] + move[1])

    # if hitting wall, stay in same state
    if not is_valid(next_state):
        return state

    return next_state

# value iteration algorithm
# performs value iteration to compute optimal value function
def value_iteration():
    global V

    iteration = 0

    while True:
        delta = 0 # track max change in this iteration
        new_V = np.copy(V) # create copy to update variable

        # loop over all states in grid
        for r in range(rows):
            for c in range(cols):
                state = (r, c)

                # skip terminal states
                if state == goal_state or state == hole_state:
                    continue

                # store value for each action
                action_values = []

                for action in actions:
                    # get next state after taking action
                    next_state = get_next_state(state, action)

                    # get reward for moving into next state
                    reward = reward_grid[next_state]

                    # bellman update equation
                    value = reward + gamma * V[next_state]
                    action_values.append(value)

                # update value funciton with best action
                new_V[state] = max(action_values)

                # update delta - largest change across states
                delta = max(delta, abs(new_V[state] - V[state]))

        # replace old value function with updated
        V = new_V
        iteration += 1

        # stop when balues converge
        if delta < theta:
            print(f"Converged in {iteration} iterations.")
            break

# extract optimal policy
'''
Extracts optimal policy from computed value function

Returns:
  policy: optimal action at each state
            G for goal, H for hole
'''
def extract_policy():
    policy = np.empty((rows, cols), dtype=str)

    # loop over all states
    for r in range(rows):
        for c in range(cols):
            state = (r, c)

            # mark terminal states
            if state == goal_state:
                policy[state] = 'G'
                continue
            if state == hole_state:
                policy[state] = 'H'
                continue

            best_action = None
            best_value = float('-inf')

            # evaluate all possible actions
            for action in actions:
                next_state = get_next_state(state, action)
                reward = reward_grid[next_state]

                # compute action value
                value = reward + gamma * V[next_state]

                if value > best_value:
                    best_value = value
                    best_action = action

            # assign best action to policy
            policy[state] = best_action

    return policy

In [11]:
# Show the learned value function and the final policy in a readable form

# run value iteration
value_iteration()

# extract policy
policy = extract_policy()

# results
print("Value Function:")
print(np.round(V, 2))

print("\nPolicy:")
print(policy)

Converged in 7 iterations.
Value Function:
[[0.43 0.52 0.62 0.73]
 [0.52 0.   0.73 0.86]
 [0.62 0.73 0.86 1.  ]
 [0.73 0.86 1.   0.  ]]

Policy:
[['D' 'R' 'D' 'D']
 ['D' 'H' 'D' 'D']
 ['D' 'D' 'D' 'D']
 ['R' 'R' 'R' 'G']]


The output above shows the value iteration algorithm solving the MDP. Convered in 7 iterations means that the algorithm repeatedly updated the value of each state until changes became very small. It reached a stable solution where further updates would not change the values by much and only took 7 iterations. The value function grid represents the long term-reward of being in that state. Each number is the expected long-term reward of being in that state. Higher values means better states which means closer to the goal. The goal state (3, 3) has value 1.0 which is the maximum reward. Value increases as getting closer to the goal which shows that the agent understands the best direction to go. The hole is treated as a terminal state and not updated, so the value of it remains 0 in the table, but it represents a −1 reward. The smooth increase toward the goal shows well-learned value landscape guiding the agent. The policy tells the best action to take from each state. D means down, R means right, H means hole which is a bad terminal state and G means goal which is the terimal state. The policy mostly moves down and right which leads to the goal at (3, 3). The agent avoids the hole at (1, 1) and the bottom row moves right toward the goal which shows the final approach. The output shows that the agent has learned an optimal policy that guides it from any state in the grid toward the goal while avoiding the hole, using value iteration to maximize long-term reward.

In [14]:
# Run at least one small experiment showing how the policy changes when you vary either the
# discount factor or the reward design

'''
Runs value iteration with specified discount factor (gamma) and returns resulting optimal policy

Parameters:
  new_gamma: discount factor to use

Returns:
  policy: optimal policy for given gamma
'''
def run_experiment(new_gamma):
    global V, gamma

    # reset values before each experiment

    # reset value function to all zeros
    V = np.zeros((rows, cols))

    # update discount factor
    gamma = new_gamma

    print(f"\nRunning with gamma = {gamma}")

    # run value iteration
    value_iteration()

    # extract optimal policy based on new values
    policy = extract_policy()

    # results
    print("Value Function:")
    print(np.round(V, 2))

    print("\nPolicy:")
    print(policy)

    return policy

# run two experiments with different gamma values

# high discount factor - values future rewards
policy_high_gamma = run_experiment(0.9)

# low discount factor - focuses on immediate rewards
policy_low_gamma = run_experiment(0.5)

# compare policies
print("Policy Comparison")

print("\nPolicy with gamma = 0.9:")
print(policy_high_gamma)

print("\nPolicy with gamma = 0.5:")
print(policy_low_gamma)


Running with gamma = 0.9
Converged in 7 iterations.
Value Function:
[[0.43 0.52 0.62 0.73]
 [0.52 0.   0.73 0.86]
 [0.62 0.73 0.86 1.  ]
 [0.73 0.86 1.   0.  ]]

Policy:
[['D' 'R' 'D' 'D']
 ['D' 'H' 'D' 'D']
 ['D' 'D' 'D' 'D']
 ['R' 'R' 'R' 'G']]

Running with gamma = 0.5
Converged in 7 iterations.
Value Function:
[[-0.05 -0.01  0.06  0.19]
 [-0.01  0.    0.19  0.46]
 [ 0.06  0.19  0.46  1.  ]
 [ 0.19  0.46  1.    0.  ]]

Policy:
[['D' 'R' 'D' 'D']
 ['D' 'H' 'D' 'D']
 ['D' 'D' 'D' 'D']
 ['R' 'R' 'R' 'G']]
Policy Comparison

Policy with gamma = 0.9:
[['D' 'R' 'D' 'D']
 ['D' 'H' 'D' 'D']
 ['D' 'D' 'D' 'D']
 ['R' 'R' 'R' 'G']]

Policy with gamma = 0.5:
[['D' 'R' 'D' 'D']
 ['D' 'H' 'D' 'D']
 ['D' 'D' 'D' 'D']
 ['R' 'R' 'R' 'G']]


The experiment compares the results of value iteration using two different discount factors, γ = 0.9 and γ = 0.5. The algorithm converges quickly in 7 iterations in both which shows that the environment is small and the value function stabilizes efficiently. For γ = 0.9, the value function shows a smooth gradient increasing toward the goal state, with values remaining relatively high even for states farther away. This shows that the agent places strong emphasis on future rewards. When γ = 0.5 the value function values are significantly lower overall, especially for states farther from the goal. This is because the agent discounts future rewards more heavily, making distant rewards less valuable.

The learned policy is exactly the same for both discount factors. The agent consistently chooses actions that move it down and to the right in both, guiding it toward the goal while avoiding the hole. This shows that the optimal path to the goal is robust to changes in the discount factor within this range. The strong positive reward structure for the goal and the small step penalty dominates the decision-making process. This results in the same optimal policy. While the discount factor significantly affects the magnitude of the value function, it does not always change the optimal policy in simple, deterministic environments like this grid world.

**Discuss why this setup is an MDP and what the learned policy is doing**

This setup satisfies all the properties of a MDP. It follows the Markov property because the next state depends only on the current state and the action taken, not on any past states or actions. All the key components of an MDP are clearly defined. The states are the positions in the grid, the actions are the possible movement directions (up, down, left, right), the transition function is deterministic based on these movements, the rewards are assigned based on the type of state (goal, hole, or regular), and a discount factor γ is used to weigh future rewards. The learned policy helps guide the agent toward the goal state, which provides a positive reward, while avoiding the hole that results in a penalty. It also minimizes the number of steps taken by incorporating a small negative reward for each move. The policy represents the optimal strategy for maximizing the total cumulative reward over time.